# Source of Volume (SoV) Analysis

## Overview
Extract SoV (Source of Volume) to understand **what customers bought BEFORE purchasing the target product**.

**Key Question:** Where are your target product purchasers coming from?

**Use Cases:**
- Identify competitive products being replaced
- Understand pre-purchase behavior patterns
- Discover cross-category switching
- Measure brand switching dynamics

**Data Source:** `cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw`

**Template Updated:** 2025/03/04  
**Python Migration:** 2026/01/07

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import databricks.sql as sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import plotly.graph_objects as go
import plotly.express as px

print("✓ Libraries imported successfully")

In [ ]:
# Load environment variables
load_dotenv(dotenv_path='../.env')

DATABRICKS_HOST = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')

if not all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]):
    raise ValueError("Missing required Databricks credentials. Please check your .env file.")

print(f"✓ Configuration loaded")

## 2. Analysis Parameters

Configure SoV analysis with pre-period (where they came from) and post-period (target product purchase).

In [ ]:
# =============================================================================
# SoV ANALYSIS PARAMETERS
# =============================================================================

# Customer Filter
customer_filter = 'cds_8007'

# Pre-Period (Source of Volume period - what they bought BEFORE)
pre_start_date = '2023-10-01'
pre_end_date = '2023-12-31'

# Post-Period (Target product purchase period)
post_start_date = '2024-01-01'
post_end_date = '2024-03-31'

# Category Filter
category_filter = 'Laundry'

# Target Condition (products purchased in POST period that we're analyzing)
# This defines which product purchases we're tracking back to find their source
target_condition = "jp_sub_brand_alter_lang_name IN ('アリエール', 'ボールド')"

# SoV Granularity (how to group the PRE-period purchases)
# Level 1: Primary grouping
sov_granularity_1 = 'jp_brand_alter_lang_name'
# Level 2: Secondary grouping
sov_granularity_2 = 'jp_sub_brand_alter_lang_name'

print("✓ Parameters configured:")
print(f"  Customer: {customer_filter}")
print(f"  Pre-period: {pre_start_date} to {pre_end_date}")
print(f"  Post-period: {post_start_date} to {post_end_date}")
print(f"  Category: {category_filter}")
print(f"  Target: {target_condition}")
print(f"  Granularity: {sov_granularity_1} → {sov_granularity_2}")

## 3. Build and Execute Query

In [ ]:
# Build SoV query
query = f"""
WITH base AS (
    SELECT
        jp_item_gtin,
        jp_prod_name,
        jp_brand_name,
        jp_brand_alter_lang_name,
        jp_category_name,
        jp_sub_category_name,
        jp_sub_category_alter_lang_name,
        jp_segment_name,
        jp_sub_segment_name,
        jp_sub_brand_name,
        jp_sub_brand_alter_lang_name,
        jp_prod_alter_lang_name,
        jp_prod_family_1_name,
        jp_prod_form_name,
        jp_size_name,
        jp_pack_size_name,
        idpos.shopper_key AS id,
        pos_unit_sales_qty AS unit,
        pos_sales_amt AS value,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date
    FROM
        cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
        LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
        LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE
        jp_category_name = '{category_filter}'
        AND idpos.data_provider_code_part = '{customer_filter}'
        AND sales_period_group_end_date_part BETWEEN '{pre_start_date}' AND '{post_end_date}'
        AND shopper.member_ind = 'Y'
),
pre AS (
    SELECT
        {sov_granularity_1} AS sov_granularity_1,
        {sov_granularity_2} AS sov_granularity_2,
        id,
        value,
        unit
    FROM base
    WHERE purchase_date BETWEEN '{pre_start_date}' AND '{pre_end_date}'
),
post AS (
    SELECT DISTINCT id
    FROM base
    WHERE {target_condition}
        AND purchase_date BETWEEN '{post_start_date}' AND '{post_end_date}'
)
SELECT
    'Post All' AS sov_granularity_1,
    'Post All' AS sov_granularity_2,
    COUNT(DISTINCT id) AS SoV_ID,
    NULL AS SoV_Value,
    NULL AS SoV_Unit
FROM post

UNION ALL

SELECT
    NVL(t2.sov_granularity_1, 'TTL') AS sov_granularity_1,
    NVL(t2.sov_granularity_2, 'TTL') AS sov_granularity_2,
    COUNT(DISTINCT t2.id) AS SoV_ID,
    ROUND(SUM(t2.value), 0) AS SoV_Value,
    ROUND(SUM(t2.unit), 0) AS SoV_Unit
FROM post AS t1
LEFT JOIN pre AS t2 ON t1.id = t2.id
WHERE t2.id IS NOT NULL
GROUP BY ROLLUP(t2.sov_granularity_1, t2.sov_granularity_2)
ORDER BY SoV_ID DESC
"""

print("✓ SQL query constructed")
print(f"  Query length: {len(query)} characters")

In [ ]:
# Execute query
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)

# Convert data types
df['SoV_ID'] = df['SoV_ID'].astype('Int64')
df['SoV_Value'] = pd.to_numeric(df['SoV_Value'], errors='coerce')
df['SoV_Unit'] = pd.to_numeric(df['SoV_Unit'], errors='coerce')

print(f"✓ Query executed successfully")
print(f"  Retrieved {len(df)} rows")
print(f"\nFirst few rows:")
df.head()

## 4. Process and Analyze Results

In [ ]:
# Separate total row and detail rows
total_purchasers = df.loc[df['sov_granularity_1'] == 'Post All', 'SoV_ID'].iloc[0]
detail_df = df[df['sov_granularity_1'] != 'Post All'].copy()

# Filter for top-level granularity (not subtotals or TTL)
top_level = detail_df[
    (detail_df['sov_granularity_2'] != 'TTL') & 
    (detail_df['sov_granularity_1'].notna())
].copy()

# Calculate percentages
top_level['pct_of_purchasers'] = (top_level['SoV_ID'] / total_purchasers * 100).round(1)
top_level['pct_of_value'] = (top_level['SoV_Value'] / top_level['SoV_Value'].sum() * 100).round(1)

# Sort by shopper count
top_level = top_level.sort_values('SoV_ID', ascending=False)

# Summary statistics
print("=" * 60)
print("SOURCE OF VOLUME ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nTarget Product Purchasers (Post-period): {total_purchasers:,}")
print(f"Matched to Pre-period: {top_level['SoV_ID'].sum():,}")
print(f"Match Rate: {(top_level['SoV_ID'].sum() / total_purchasers * 100):.1f}%")
print(f"\nTop 5 Source Brands:")
for idx, row in top_level.head(5).iterrows():
    print(f"  {row['sov_granularity_1']:30s} - {row['SoV_ID']:6,} shoppers ({row['pct_of_purchasers']:5.1f}%)")

top_level.head(10)

## 5. Visualizations

In [ ]:
# Sankey diagram for Source of Volume flow
# Prepare data for Sankey (top 15 sources + others)
top_n = 15
sankey_data = top_level.head(top_n).copy()

# Add "Others" category for remaining brands
others_count = top_level[top_n:]['SoV_ID'].sum() if len(top_level) > top_n else 0
if others_count > 0:
    others_row = pd.DataFrame({
        'sov_granularity_1': ['Others'],
        'sov_granularity_2': ['Others'],
        'SoV_ID': [others_count],
        'pct_of_purchasers': [(others_count / total_purchasers * 100)]
    })
    sankey_data = pd.concat([sankey_data, others_row], ignore_index=True)

# Create node labels
source_labels = sankey_data['sov_granularity_1'].tolist()
target_label = f"Target Product\n({total_purchasers:,} shoppers)"
all_labels = source_labels + [target_label]

# Create links
source_indices = list(range(len(source_labels)))
target_index = len(source_labels)
link_values = sankey_data['SoV_ID'].tolist()

# Color scheme
colors = px.colors.qualitative.Set3[:len(source_labels)]

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_labels,
        color=['lightblue'] * len(source_labels) + ['#FF6B6B']
    ),
    link=dict(
        source=source_indices,
        target=[target_index] * len(source_indices),
        value=link_values,
        color=['rgba(135, 206, 250, 0.4)'] * len(source_indices)
    )
)])

fig.update_layout(
    title=f"Source of Volume Flow: {category_filter} Category<br>Pre-period → Target Product (Post-period)",
    font_size=12,
    height=600
)

fig.show()

In [ ]:
# Bar chart for top source brands
fig = px.bar(
    top_level.head(15),
    x='SoV_ID',
    y='sov_granularity_1',
    orientation='h',
    title=f"Top 15 Source Brands for Target Product",
    labels={'SoV_ID': 'Number of Shoppers', 'sov_granularity_1': 'Brand'},
    text='SoV_ID',
    color='pct_of_purchasers',
    color_continuous_scale='Blues'
)

fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    height=500,
    showlegend=False,
    coloraxis_colorbar_title="% of Target<br>Purchasers"
)

fig.show()

## 6. Data Export

In [ ]:
# Export to Excel (optional)
export_file = f"sov_analysis_{category_filter}_{post_start_date}.xlsx"

with pd.ExcelWriter(export_file, engine='openpyxl') as writer:
    # Summary sheet
    summary_df = pd.DataFrame({
        'Metric': ['Total Target Purchasers (Post)', 'Matched to Pre-period', 'Match Rate %'],
        'Value': [total_purchasers, top_level['SoV_ID'].sum(), 
                  f"{(top_level['SoV_ID'].sum() / total_purchasers * 100):.1f}%"]
    })
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Top sources
    top_level.to_excel(writer, sheet_name='Top_Sources', index=False)
    
    # Full detail
    df.to_excel(writer, sheet_name='Full_Data', index=False)

print(f"✓ Data exported to: {export_file}")